# FlixBus Ticket Explorer

Bu notebook `flixbus_finder.py` modülündeki fonksiyonları kullanarak bilet aramayı gösteriyor.

**Nasıl çalışır:**
1. `find_city` -- şehri önce `flixbus_europe.db` içinde arar (isim veya alias). Yoksa autocomplete API'ye sorar, öğrenir ve kaydeder.
2. `search_tickets` -- `global.api.flixbus.com/search/service/v4/search` endpoint'inden kalkış/varış saati, süre, fiyat, koltuk döner.
3. `search_range` -- verilen tarih aralığında her gün ayrı istek atar, tümünü birleştirir ve fiyata göre sıralar.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from flixbus_finder import (
    init_db,
    find_city,
    search_tickets,
    search_range,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

conn = init_db()  # flixbus_europe.db baglanir, yoksa olusturur
print('DB baglantisi hazir.')

DB baglantisi hazir.


## 1. Sehir ID bul

`find_city` once DB'ye bakar; bulamazsa API'ye sorar, official ismi ve alias'i kaydeder.
Artik Almanya disindaki Avrupa sehirleri de destekleniyor.

In [2]:
berlin  = find_city(conn, 'Berlin')
munich  = find_city(conn, 'Munich')
venice  = find_city(conn, 'Venice')
hamburg = find_city(conn, 'Hamburg')

print(f'Berlin  -> {berlin}')
print(f'Munich  -> {munich}')
print(f'Venice  -> {venice}')
print(f'Hamburg -> {hamburg}')

  🔍 'Berlin' DB'de yok, API'ye danışılıyor...
  🔍 'Hamburg' DB'de yok, API'ye danışılıyor...
Berlin  -> ('40d8f682-8646-11e6-9066-549f350fcb0c', 'Berlin')
Munich  -> ('40d901a5-8646-11e6-9066-549f350fcb0c', 'Munich')
Venice  -> ('40dea03b-8646-11e6-9066-549f350fcb0c', 'Venice')
Hamburg -> ('40d91e53-8646-11e6-9066-549f350fcb0c', 'Hamburg')


## 2. Tek gun bilet arama

In [4]:
tickets = search_tickets(berlin[0], munich[0], '01.06.2026')
print(f'{len(tickets)} sefer bulundu.')

df = pd.DataFrame(tickets)
df

15 sefer bulundu.


,tarih,kalkış,varış,süre,fiyat,koltuk,url
0,01.06.2026,2026-06-01T17:20:00+02:00,2026-06-02T00:40:00+02:00,7s 20dk,20.49,44,https://shop.global.flixbus.com/search?departureCity=40d8f682-8646-11e6-9066...
1,01.06.2026,2026-06-01T13:50:00+02:00,2026-06-01T22:00:00+02:00,8s 10dk,20.99,64,https://shop.global.flixbus.com/search?departureCity=40d8f682-8646-11e6-9066...
2,01.06.2026,2026-06-01T14:20:00+02:00,2026-06-01T22:00:00+02:00,7s 40dk,20.99,69,https://shop.global.flixbus.com/search?departureCity=40d8f682-8646-11e6-9066...
3,01.06.2026,2026-06-01T22:20:00+02:00,2026-06-02T05:30:00+02:00,7s 10dk,23.99,74,https://shop.global.flixbus.com/search?departureCity=40d8f682-8646-11e6-9066...
4,01.06.2026,2026-06-01T22:20:00+02:00,2026-06-02T05:55:00+02:00,7s 35dk,23.99,74,https://shop.global.flixbus.com/search?departureCity=40d8f682-8646-11e6-9066...
5,01.06.2026,2026-06-01T21:00:00+02:00,2026-06-02T04:45:00+02:00,7s 45dk,24.49,70,https://shop.global.flixbus.com/search?departureCity=40d8f682-8646-11e6-9066...
6,01.06.2026,2026-06-01T23:20:00+02:00,2026-06-02T07:00:00+02:00,7s 40dk,24.49,47,https://shop.global.flixbus.com/search?departureCity=40d8f682-8646-11e6-9066...
7,01.06.2026,2026-06-01T23:20:00+02:00,2026-06-02T07:25:00+02:00,8s 5dk,24.49,47,https://shop.global.flixbus.com/search?departureCity=40d8f682-8646-11e6-9066...
8,01.06.2026,2026-06-01T19:10:00+02:00,2026-06-02T05:30:00+02:00,10s 20dk,29.98,49,https://shop.global.flixbus.com/search?departureCity=40d8f682-8646-11e6-9066...
9,01.06.2026,2026-06-01T19:57:00+02:00,2026-06-02T04:45:00+02:00,8s 48dk,29.98,70,https://shop.global.flixbus.com/search?departureCity=40d8f682-8646-11e6-9066...


### Fiyat istatistikleri

In [5]:
df['fiyat'].describe()

count    15.000000
mean     26.919333
std       4.832965
min      20.490000
25%      23.990000
50%      24.490000
75%      30.230000
max      36.990000
Name: fiyat, dtype: float64

### En ucuz 5 sefer

In [6]:
df.nsmallest(5, 'fiyat')[['kalkis', 'varis', 'sure', 'fiyat', 'koltuk']]
# Sutun isimlerine dikkat: API'den gelen Turkce karakterleri iceriyor
df.nsmallest(5, 'fiyat')[['kalkış', 'varış', 'süre', 'fiyat', 'koltuk']]

KeyError: "['kalkis', 'varis', 'sure'] not in index"

### Kalkis saatine gore fiyat dagilimi

In [7]:
df2 = df.copy()
df2['kalkış_dt'] = pd.to_datetime(df2['kalkış'])
df2['kalkış_saat'] = df2['kalkış_dt'].dt.hour

def zaman_dilimi(saat):
    if saat < 12:
        return 'Sabah (00-11)'
    elif saat < 18:
        return 'Ogle sonrasi (12-17)'
    else:
        return 'Aksam (18-23)'

df2['dilim'] = df2['kalkış_saat'].apply(zaman_dilimi)
df2.groupby('dilim')['fiyat'].agg(['min', 'mean', 'max', 'count']).round(2)

,min,mean,max,count
dilim,,,,
Aksam (18-23),23.99,27.38,31.98,10
Ogle sonrasi (12-17),20.49,23.24,30.48,4
Sabah (00-11),36.99,36.99,36.99,1


## 3. Tarih araligi arama

`search_range` her gun ayri istek atar, tum seferleri birlestirir.

In [ ]:
all_tickets = search_range(berlin[0], munich[0], '01-06-2026', '05-06-2026')
print(f'Toplam {len(all_tickets)} sefer.')

df_range = pd.DataFrame(all_tickets)
df_range.head(10)

### Her gun icin en ucuz bilet

In [ ]:
df_range.groupby('tarih')['fiyat'].min().reset_index().rename(columns={'fiyat': 'en_ucuz_EUR'})

### Her gun sefer sayisi ve fiyat ozeti

In [ ]:
df_range.groupby('tarih')['fiyat'].agg(['count', 'min', 'mean', 'max']).round(2).rename(
    columns={'count': 'sefer_sayisi', 'min': 'en_ucuz', 'mean': 'ortalama', 'max': 'en_pahali'}
)

## 4. Farkli guzergah dene

Artik Avrupa genelinde herhangi bir sehir calisir. Italya, Fransa, vs.

In [8]:
from_city = find_city(conn, 'Venice')
to_city   = find_city(conn, 'Munich')
tarih     = '01.06.2026'

print(f'Guzergah: {from_city[1]} -> {to_city[1]}')

results = search_tickets(from_city[0], to_city[0], tarih)
pd.DataFrame(results)[['kalkış', 'varış', 'süre', 'fiyat', 'koltuk']]

Guzergah: Venice -> Munich


,kalkış,varış,süre,fiyat,koltuk
0,2026-06-01T21:00:00+02:00,2026-06-02T04:35:00+02:00,7s 35dk,41.99,41
1,2026-06-01T21:20:00+02:00,2026-06-02T04:35:00+02:00,7s 15dk,41.99,41
2,2026-06-01T21:55:00+02:00,2026-06-02T06:30:00+02:00,8s 35dk,41.99,71
3,2026-06-01T22:15:00+02:00,2026-06-02T06:30:00+02:00,8s 15dk,41.99,71
4,2026-06-01T08:00:00+02:00,2026-06-01T16:30:00+02:00,8s 30dk,43.99,43
5,2026-06-01T08:20:00+02:00,2026-06-01T16:30:00+02:00,8s 10dk,43.99,43
6,2026-06-01T17:20:00+02:00,2026-06-02T01:35:00+02:00,8s 15dk,43.99,43
7,2026-06-01T17:40:00+02:00,2026-06-02T01:35:00+02:00,7s 55dk,43.99,43
8,2026-06-01T12:55:00+02:00,2026-06-01T21:30:00+02:00,8s 35dk,45.99,66
9,2026-06-01T13:15:00+02:00,2026-06-01T21:30:00+02:00,8s 15dk,45.99,66


## 5. DB'deki tum sehirler

Daha once aranip kaydedilmis sehirler (alias bilgisiyle birlikte).

In [ ]:
rows = conn.execute('SELECT name, id, country, search_terms FROM cities ORDER BY name').fetchall()
df_cities = pd.DataFrame(rows, columns=['sehir', 'id', 'ulke', 'alias'])
print(f'Toplam {len(df_cities)} sehir DB de kayitli.')
df_cities